# 04 - Transformer sentiment classifier and its explanations  (Req 2b, Req 7)

**Goal:** classify tweets as negative / neutral / positive with a transformer, measure how well it does on the hand-labelled tweets, and show *which words* led to each label.

Two transformer models:

| | Model | Training | Classes |
|---|---|---|---|
| **A** (main) | `cardiffnlp/twitter-roberta-base-sentiment-latest` | none by us - already fine-tuned on TweetEval | negative / neutral / positive |
| **B** | `distilbert-base-uncased` | fine-tuned by us on 20k emoticon-labelled tweets | negative / positive |

The work is done by `src/04`, `src/05` and `src/06`; this notebook runs them and shows the results.

**Run this on a GPU.** On Google Colab: *Runtime -> Change runtime type -> T4 GPU*, then *Run all* (about 10 minutes). On a laptop CPU, set `QUICK = True` below.

In [ ]:
import os, sys, json, subprocess
BRANCH = 'transformer-classifier'   # the GitHub branch that has src/04-06 (use 'main' once merged)
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not os.path.exists('/content/NamPulse'):
    subprocess.run(['git', 'clone', '-b', BRANCH, 'https://github.com/AKChumba/NamPulse.git', '/content/NamPulse'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', '/content/NamPulse/requirements.txt'], check=True)
if IN_COLAB:
    os.chdir('/content/NamPulse/notebooks')
sys.path.insert(0, '../src')

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, Image, display
import torch
pd.set_option('display.max_colwidth', 110)
print('GPU available:', torch.cuda.is_available())

QUICK = not torch.cuda.is_available()   # True = small runs that finish on a laptop CPU
RERUN = True                            # False = just load the saved reports

## A. Pretrained Twitter-RoBERTa (no training)
Scores every tweet and evaluates on the 493 hand-labelled tweets (3 classes), then on the 354 positive/negative ones (to compare with the classical model).

In [ ]:
if RERUN:
    args = '--limit 5000' if QUICK else ''
    !python ../src/04_transformer_pretrained.py {args}
print(open('../reports/transformer_pretrained_report.txt').read())

In [ ]:
display(Image('../reports/transformer_pretrained_confusion.png', width=380))

## B. DistilBERT fine-tuned on our emoticon labels
Train/validation come from the emoticon-labelled tweets (each distinct text used once, nothing that is also in the test set). Test = the hand-labelled positive/negative tweets.

In [ ]:
if RERUN:
    args = '--n-train 4000 --epochs 1' if QUICK else ''
    !python ../src/05_transformer_finetune.py {args}
print(open('../reports/transformer_finetune_report.txt').read())
display(Image('../reports/transformer_finetune_curve.png', width=380))

## Comparison on the same hand-labelled test tweets

In [ ]:
a = json.load(open('../reports/transformer_pretrained_metrics.json'))
b = json.load(open('../reports/transformer_finetune_metrics.json'))
rows = [{'model': 'A: Twitter-RoBERTa (pretrained)', 'pos/neg accuracy': a['manual_binary']['accuracy'],
         'pos/neg macro-F1': a['manual_binary']['macro_f1'], '3-class accuracy': a['manual_3class']['accuracy'],
         '3-class macro-F1': a['manual_3class']['macro_f1'], 'our training data': 'none'},
        {'model': 'B: DistilBERT (fine-tuned by us)', 'pos/neg accuracy': b['manual_binary']['accuracy'],
         'pos/neg macro-F1': b['manual_binary']['macro_f1'], '3-class accuracy': None, '3-class macro-F1': None,
         'our training data': f"{b['n_train']:,} tweets, {b['epochs']} epoch(s)"}]
# The classical model (Req 2a) - add its numbers here once Jona's metrics file is in the repo
classical = '../reports/classical_metrics.json'
if os.path.exists(classical):
    c = json.load(open(classical))
    rows.append({'model': 'Classical: TF-IDF + logistic regression', **c})
pd.DataFrame(rows).set_index('model')

## C. Explanations: which words decided the label?  (Req 7)
Integrated Gradients gives every word a score: **green** pushed the tweet towards the predicted label, **red** pushed against it. `src/06` also checks the scores against a simple leave-one-word-out test.

In [ ]:
if RERUN:
    args = '--limit 100' if QUICK else ''
    !python ../src/06_transformer_explain.py {args}
print(open('../reports/transformer_explain_report.txt').read())

In [ ]:
html = open('../reports/transformer_explanations.html', encoding='utf-8').read()
display(HTML(html[html.find('<h2>'):html.rfind('</body>')]))

## D. Try it on your own text (including code-switched Namibian examples, Req 6)
The model was trained on English tweets, so watch what it does with Afrikaans or Oshiwambo words.

In [ ]:
from nampulse_transformer import load_model, explain, to_html
tok, model = load_model('cardiffnlp/twitter-roberta-base-sentiment-latest')
examples = [
    'Water is weer af in Windhoek, this is unacceptable, ons betaal vir niks',
    'Die diens was baie sleg and the staff were rude to everyone',
    'NamWater fixed the pipe so quickly, baie dankie!',
    'The new policy starts next month.',
]
for t in examples:
    display(HTML('<p>' + to_html(explain(t, tok, model)) + '</p>'))

## Save the results
On Colab this zips every file the three scripts wrote and downloads it. Unzip it into your local NamPulse folder and commit it. The fine-tuned model in `models/` is left out because it is too big for git.

In [ ]:
import shutil, glob
keep = (glob.glob('../reports/transformer_*') + glob.glob('../data/processed/transformer_*')
        + glob.glob('../data/processed/finetuned_test_predictions.csv'))
shutil.rmtree('/tmp/results', ignore_errors=True)
for f in keep:
    dest = os.path.join('/tmp/results', os.path.relpath(f, '..'))
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    shutil.copy(f, dest)
shutil.make_archive('/tmp/nampulse_transformer_results', 'zip', '/tmp/results')
print(len(keep), 'files zipped:')
for f in sorted(keep):
    print('  ', os.path.relpath(f, '..'))
if IN_COLAB:
    from google.colab import files
    files.download('/tmp/nampulse_transformer_results.zip')

## What we found
*(Write this part in our own words after running the notebook - the brief requires the interpretation to be ours.)*

- Model A vs model B vs the classical model: ...
- Where the transformer goes wrong (see the Mistakes examples): ...
- What the explanations show / one output traced back to its words: ...
- Code-switched and non-English text: ...
- Bias / data-quality risk we observed: ...